# EchoMind Prosody v5 — RESUME CACHE ONLY (sem reparação agressiva)

Este notebook **não volta a correr WavLM/Transformers**.  
Ele só carrega os ficheiros já gerados em `prosody_outputs_v5` e corre os modelos clássicos para veres a qualidade.

**Começa por correr a partir da célula 1.**  
Não há célula com `os.kill(...)` nem reinstalação agressiva de NumPy.


## 1) Imports, Drive e configuração

Se esta célula falhar logo nos imports de `numpy`, `pandas` ou `sklearn`, então o ambiente ainda está partido. Nesse caso, cria um runtime limpo no Colab e volta a abrir este notebook.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from collections import Counter
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import StratifiedGroupKFold, GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    classification_report, confusion_matrix
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("sklearn OK")

Mounted at /content/drive
numpy: 2.0.2
pandas: 2.2.2
sklearn OK


## 2) Verificar ficheiros da pasta `prosody_outputs_v5`

Esta parte confirma se tens os ficheiros necessários no Drive.  
Se algum nome for diferente, a célula tenta encontrar automaticamente ficheiros parecidos.


In [2]:
OUTPUT_DIR = Path("/content/drive/MyDrive/prosody_outputs_v5")
assert OUTPUT_DIR.exists(), f"Pasta não encontrada: {OUTPUT_DIR}"

print("Ficheiros encontrados em prosody_outputs_v5:")
for p in sorted(OUTPUT_DIR.iterdir()):
    if p.is_file():
        print("-", p.name, "|", round(p.stat().st_size / 1024 / 1024, 2), "MB")

def pick_file(candidates, glob_patterns=None, required=True):
    for name in candidates:
        p = OUTPUT_DIR / name
        if p.exists():
            return p
    if glob_patterns:
        matches = []
        for pat in glob_patterns:
            matches.extend(sorted(OUTPUT_DIR.glob(pat)))
        if matches:
            return matches[0]
    if required:
        raise FileNotFoundError(f"Não encontrei nenhum destes ficheiros: {candidates} / {glob_patterns}")
    return None

manual_path = pick_file(
    ["manual_features_v5.csv", "manual_features.csv"],
    ["*manual*features*.csv", "*features*v5*.csv"]
)

manual_cols_path = pick_file(
    ["manual_feature_columns_v5.json", "manual_feature_cols_v5.json", "manual_feature_columns.json"],
    ["*manual*column*.json", "*feature*col*.json"]
)

ssl_npz_path = pick_file(
    ["ssl_embeddings_v5_wavlm_base.npz", "ssl_embeddings_wavlm_base.npz", "ssl_embeddings.npz"],
    ["*ssl*wavlm*.npz", "*embedding*.npz"]
)

ssl_meta_path = pick_file(
    ["ssl_embeddings_v5_wavlm_base_meta.csv", "ssl_embeddings_wavlm_base_meta.csv", "ssl_meta.csv"],
    ["*ssl*meta*.csv", "*embedding*meta*.csv"]
)

print("\nUsando:")
print("manual_path:", manual_path.name)
print("manual_cols_path:", manual_cols_path.name)
print("ssl_npz_path:", ssl_npz_path.name)
print("ssl_meta_path:", ssl_meta_path.name)

Ficheiros encontrados em prosody_outputs_v5:
- class_priors_v5.json | 0.0 MB
- manual_feature_columns_v5.json | 0.03 MB
- manual_features_v5.csv | 276.95 MB
- metadata_v5_clean.csv | 1.64 MB
- ssl_embedding_failures_v5.csv | 0.0 MB
- ssl_embeddings_v5_wavlm_base.npz | 68.61 MB
- ssl_embeddings_v5_wavlm_base_meta.csv | 0.9 MB
- ssl_feature_columns_v5.json | 0.02 MB

Usando:
manual_path: manual_features_v5.csv
manual_cols_path: manual_feature_columns_v5.json
ssl_npz_path: ssl_embeddings_v5_wavlm_base.npz
ssl_meta_path: ssl_embeddings_v5_wavlm_base_meta.csv


## 3) Carregar features manuais e embeddings SSL já existentes

Aqui não há `transformers`, não há WavLM, não há extração nova.


In [3]:
with open(manual_cols_path, "r", encoding="utf-8") as f:
    manual_feature_cols = json.load(f)

manual_df = pd.read_csv(manual_path)
ssl_meta = pd.read_csv(ssl_meta_path)

npz = np.load(ssl_npz_path)
print("Chaves no .npz:", list(npz.keys()))

if "X_ssl" in npz:
    X_ssl = npz["X_ssl"]
elif "embeddings" in npz:
    X_ssl = npz["embeddings"]
elif len(npz.files) == 1:
    X_ssl = npz[npz.files[0]]
else:
    raise KeyError(f"Não sei qual chave usar no NPZ. Chaves: {npz.files}")

print("manual_df:", manual_df.shape)
print("n manual_feature_cols:", len(manual_feature_cols))
print("ssl_meta:", ssl_meta.shape)
print("X_ssl:", X_ssl.shape)

required_base_cols = ["path", "filename", "label", "speaker", "source_dataset"]
missing = [c for c in required_base_cols if c not in manual_df.columns]
if missing:
    raise ValueError(f"Faltam colunas em manual_df: {missing}. Colunas existentes: {manual_df.columns.tolist()[:30]}")

Chaves no .npz: ['X_ssl']
manual_df: (12788, 1255)
n manual_feature_cols: 1250
ssl_meta: (12788, 4)
X_ssl: (12788, 1536)


## 4) Alinhar tabelas

Reconstrói `aligned_df`, `X_manual`, `X_ssl_aligned`, `y`, `groups` e `sources`.


In [4]:
EMOTIONS = ["joy", "sadness", "surprise", "fear", "anger", "disgust", "neutral"]

def align_tables_from_cache():
    base_cols = ["path", "filename", "label", "speaker", "source_dataset"]

    data = manual_df[base_cols].drop_duplicates("path").copy()

    # Features manuais
    feature_table = manual_df[["path"] + manual_feature_cols].drop_duplicates("path")
    data = data.merge(feature_table, on="path", how="inner")

    # Índice SSL
    sm = ssl_meta.copy()
    if "ssl_idx" not in sm.columns:
        sm = sm.reset_index().rename(columns={"index": "ssl_idx"})

    if "path" not in sm.columns:
        raise ValueError(f"ssl_meta não tem coluna 'path'. Colunas: {sm.columns.tolist()}")

    data = data.merge(sm[["path", "ssl_idx"]].drop_duplicates("path"), on="path", how="inner")

    Xman = data[manual_feature_cols].to_numpy(dtype=np.float32)
    Xssl_aligned = X_ssl[data["ssl_idx"].to_numpy()].astype(np.float32)

    return data, Xman, Xssl_aligned

aligned_df, X_manual, X_ssl_aligned = align_tables_from_cache()

y_labels = aligned_df["label"].astype(str).to_numpy()
groups = aligned_df["speaker"].astype(str).to_numpy()
sources = aligned_df["source_dataset"].astype(str).to_numpy()

# Caso apareça alguma label fora da lista, avisa logo.
unknown = sorted(set(y_labels) - set(EMOTIONS))
if unknown:
    raise ValueError(f"Labels desconhecidas: {unknown}. Ajusta a lista EMOTIONS.")

label_to_idx = {e: i for i, e in enumerate(EMOTIONS)}
idx_to_label = {i: e for e, i in label_to_idx.items()}
y = np.array([label_to_idx[l] for l in y_labels], dtype=int)

print("aligned:", aligned_df.shape)
print("X_manual:", X_manual.shape)
print("X_ssl_aligned:", X_ssl_aligned.shape)
print("y:", y.shape)
print("speakers:", len(np.unique(groups)))
print("\nDistribuição por emoção:")
print(pd.Series(y_labels).value_counts())
print("\nDistribuição por source_dataset:")
print(pd.Series(sources).value_counts())

aligned: (12788, 1256)
X_manual: (12788, 1250)
X_ssl_aligned: (12788, 1536)
y: (12788,)
speakers: 118

Distribuição por emoção:
anger       2167
joy         2167
sadness     2166
fear        2047
disgust     1863
neutral     1786
surprise     592
Name: count, dtype: int64

Distribuição por source_dataset:
CREMA-D    7441
TESS       2799
RAVDESS    2068
SAVEE       480
Name: count, dtype: int64


## 5) Definir feature sets

Testa manual, SSL e manual+SSL.


In [5]:
feature_matrices = {
    "manual": X_manual,
    "ssl_wavlm": X_ssl_aligned,
    "manual_plus_ssl": np.hstack([X_manual, X_ssl_aligned]).astype(np.float32),
}

for name, X in feature_matrices.items():
    print(name, X.shape)

manual (12788, 1250)
ssl_wavlm (12788, 1536)
manual_plus_ssl (12788, 2786)


## 6) Funções de avaliação

Usa validação cruzada com grupos por `speaker`, para evitar leakage entre o mesmo falante no treino e validação.


In [6]:
RANDOM_STATE = 42
N_SPLITS = 5

n_groups = len(np.unique(groups))
n_min_class = pd.Series(y).value_counts().min()
effective_splits = int(min(N_SPLITS, n_groups, n_min_class))
if effective_splits < 2:
    raise ValueError("Não há grupos/classes suficientes para validação cruzada.")

try:
    cv = StratifiedGroupKFold(n_splits=effective_splits, shuffle=True, random_state=RANDOM_STATE)
    splits = list(cv.split(np.zeros(len(y)), y, groups))
    print(f"Usando StratifiedGroupKFold com {effective_splits} folds.")
except Exception as e:
    print("StratifiedGroupKFold falhou, vou usar GroupKFold. Motivo:", repr(e))
    cv = GroupKFold(n_splits=effective_splits)
    splits = list(cv.split(np.zeros(len(y)), y, groups))
    print(f"Usando GroupKFold com {effective_splits} folds.")

def make_models():
    return {
        "logreg_balanced": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                max_iter=4000,
                class_weight="balanced",
                C=1.0,
                solver="lbfgs",
                multi_class="auto",
                random_state=RANDOM_STATE
            ))
        ]),
        "linear_svc_calibrated": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", CalibratedClassifierCV(
                estimator=LinearSVC(
                    C=0.5,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                    max_iter=5000
                ),
                cv=3
            ))
        ]),
        "rbf_svc_balanced": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", SVC(
                C=3.0,
                gamma="scale",
                kernel="rbf",
                class_weight="balanced",
                probability=True,
                random_state=RANDOM_STATE
            ))
        ]),
        "extra_trees": ExtraTreesClassifier(
            n_estimators=600,
            max_features="sqrt",
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        "random_forest": RandomForestClassifier(
            n_estimators=500,
            max_features="sqrt",
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        "hist_gradient_boosting": HistGradientBoostingClassifier(
            max_iter=300,
            learning_rate=0.05,
            random_state=RANDOM_STATE
        )
    }

def get_proba_or_scores(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)
    if hasattr(model, "decision_function"):
        s = model.decision_function(X)
        if s.ndim == 1:
            s = np.vstack([-s, s]).T
        # softmax simples
        s = s - s.max(axis=1, keepdims=True)
        exp = np.exp(s)
        return exp / exp.sum(axis=1, keepdims=True)
    pred = model.predict(X)
    P = np.zeros((len(pred), len(EMOTIONS)), dtype=float)
    P[np.arange(len(pred)), pred] = 1.0
    return P

def evaluate_cv(model_name, model, X, y, groups, feature_set):
    oof_pred = np.zeros_like(y)
    oof_proba = np.zeros((len(y), len(EMOTIONS)), dtype=float)
    fold_rows = []

    for fold, (tr, va) in enumerate(splits, start=1):
        model.fit(X[tr], y[tr])
        pred = model.predict(X[va])
        proba = get_proba_or_scores(model, X[va])

        # alinhar proba caso o classificador só veja algumas classes no treino
        if proba.shape[1] != len(EMOTIONS):
            fixed = np.zeros((len(va), len(EMOTIONS)), dtype=float)
            classes = getattr(model[-1] if isinstance(model, Pipeline) else model, "classes_", None)
            if classes is not None:
                for j, c in enumerate(classes):
                    fixed[:, int(c)] = proba[:, j]
                proba = fixed
            else:
                proba = np.eye(len(EMOTIONS))[pred]

        oof_pred[va] = pred
        oof_proba[va] = proba

        fold_rows.append({
            "feature_set": feature_set,
            "model": model_name,
            "fold": fold,
            "n_val": len(va),
            "accuracy": accuracy_score(y[va], pred),
            "balanced_accuracy": balanced_accuracy_score(y[va], pred),
            "f1_macro": f1_score(y[va], pred, average="macro"),
            "f1_weighted": f1_score(y[va], pred, average="weighted"),
        })

    overall = {
        "feature_set": feature_set,
        "model": model_name,
        "accuracy": accuracy_score(y, oof_pred),
        "balanced_accuracy": balanced_accuracy_score(y, oof_pred),
        "f1_macro": f1_score(y, oof_pred, average="macro"),
        "f1_weighted": f1_score(y, oof_pred, average="weighted"),
    }

    return overall, pd.DataFrame(fold_rows), oof_pred, oof_proba

Usando StratifiedGroupKFold com 5 folds.


## 7) Correr benchmark

Esta célula pode demorar alguns minutos.


In [ ]:
import time

results = []
fold_tables = []
oof_store = {}

for fs_name, X in feature_matrices.items():
    print("\n" + "=" * 90)
    print("FEATURE SET:", fs_name, X.shape)
    print("=" * 90)

    models = make_models()
    for model_name, model in models.items():
        t0 = time.time()
        print("A treinar:", model_name, end=" ... ")
        try:
            overall, fold_df, oof_pred, oof_proba = evaluate_cv(model_name, model, X, y, groups, fs_name)
            overall["elapsed_sec"] = round(time.time() - t0, 2)
            results.append(overall)
            fold_tables.append(fold_df)
            oof_store[(fs_name, model_name)] = {
                "pred": oof_pred,
                "proba": oof_proba,
            }
            print("OK | f1_macro =", round(overall["f1_macro"], 4))
        except Exception as e:
            print("ERRO:", repr(e))
            results.append({
                "feature_set": fs_name,
                "model": model_name,
                "accuracy": np.nan,
                "balanced_accuracy": np.nan,
                "f1_macro": np.nan,
                "f1_weighted": np.nan,
                "elapsed_sec": round(time.time() - t0, 2),
                "error": repr(e)
            })

results_df = pd.DataFrame(results).sort_values("f1_macro", ascending=False)
folds_df = pd.concat(fold_tables, ignore_index=True) if fold_tables else pd.DataFrame()

display(results_df)
results_path = OUTPUT_DIR / "prosody_model_benchmark_v5_RESUME_NO_REPAIR.csv"
results_df.to_csv(results_path, index=False)
print("Guardado:", results_path)


FEATURE SET: manual (12788, 1250)
A treinar: logreg_balanced ... OK | f1_macro = 0.4778
A treinar: linear_svc_calibrated ... OK | f1_macro = 0.4353
A treinar: rbf_svc_balanced ... OK | f1_macro = 0.5223
A treinar: extra_trees ... OK | f1_macro = 0.4718
A treinar: random_forest ... OK | f1_macro = 0.4665
A treinar: hist_gradient_boosting ... OK | f1_macro = 0.5209

FEATURE SET: ssl_wavlm (12788, 1536)
A treinar: logreg_balanced ... OK | f1_macro = 0.5604
A treinar: linear_svc_calibrated ... OK | f1_macro = 0.4969
A treinar: rbf_svc_balanced ... OK | f1_macro = 0.6152
A treinar: extra_trees ... OK | f1_macro = 0.4851
A treinar: random_forest ... OK | f1_macro = 0.4885
A treinar: hist_gradient_boosting ... OK | f1_macro = 0.5456

FEATURE SET: manual_plus_ssl (12788, 2786)
A treinar: logreg_balanced ... OK | f1_macro = 0.577
A treinar: linear_svc_calibrated ... 

In [ ]:
# ============================================================
# 7B. Guardar imediatamente os resultados parciais já obtidos
#     Versão corrigida: define OUTPUT_DIR se não existir
# ============================================================

from pathlib import Path
import os
import joblib
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1) Recuperar / definir OUTPUT_DIR
# ------------------------------------------------------------

if "OUTPUT_DIR" not in globals():
    cwd = Path.cwd()
    candidates = []

    # procurar a pasta prosody_outputs_v5 na pasta atual e nos pais
    for base in [cwd] + list(cwd.parents):
        p = base / "prosody_outputs_v5"
        if p.exists():
            candidates.append(p)

    if candidates:
        OUTPUT_DIR = candidates[0]
        print("OUTPUT_DIR encontrado automaticamente:", OUTPUT_DIR)
    else:
        # fallback: cria/usa prosody_outputs_v5 na pasta atual
        OUTPUT_DIR = cwd / "prosody_outputs_v5"
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        print("OUTPUT_DIR não existia. Criado em:", OUTPUT_DIR)
else:
    OUTPUT_DIR = Path(OUTPUT_DIR)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    print("OUTPUT_DIR já existia:", OUTPUT_DIR)

# ------------------------------------------------------------
# 2) Verificar se ainda tens os resultados em memória
# ------------------------------------------------------------

if "results" not in globals():
    raise RuntimeError(
        "Não encontro a variável results em memória. "
        "Isto pode significar que o kernel foi reiniciado ou que a célula do treino não ficou ativa."
    )

if "oof_store" not in globals():
    raise RuntimeError(
        "Não encontro a variável oof_store em memória. "
        "Isto pode significar que o kernel foi reiniciado ou que a célula do treino não ficou ativa."
    )

if len(results) == 0:
    raise RuntimeError("A lista results está vazia. Não há resultados parciais para guardar.")

# ------------------------------------------------------------
# 3) Guardar checkpoint dos resultados parciais
# ------------------------------------------------------------

CHECKPOINT_PATH = OUTPUT_DIR / "prosody_resume_checkpoint_v5_NO_REPAIR.joblib"
PARTIAL_RESULTS_PATH = OUTPUT_DIR / "prosody_model_benchmark_v5_RESUME_NO_REPAIR_PARTIAL.csv"
PARTIAL_FOLDS_PATH = OUTPUT_DIR / "prosody_folds_v5_RESUME_NO_REPAIR_PARTIAL.csv"

results_df = pd.DataFrame(results)

if "f1_macro" in results_df.columns:
    results_df = results_df.sort_values(
        "f1_macro",
        ascending=False,
        na_position="last"
    ).reset_index(drop=True)

fold_tables = globals().get("fold_tables", [])
folds_df = pd.concat(fold_tables, ignore_index=True) if fold_tables else pd.DataFrame()

results_df.to_csv(PARTIAL_RESULTS_PATH, index=False)
folds_df.to_csv(PARTIAL_FOLDS_PATH, index=False)

joblib.dump(
    {
        "results": results,
        "fold_tables": fold_tables,
        "oof_store": oof_store,
    },
    CHECKPOINT_PATH
)

print("\nCheckpoint guardado em:", CHECKPOINT_PATH)
print("Resultados parciais guardados em:", PARTIAL_RESULTS_PATH)
print("Folds parciais guardados em:", PARTIAL_FOLDS_PATH)

display(results_df)

NameError: name 'OUTPUT_DIR' is not defined

In [6]:
# ============================================================
# RECOVERY FINAL SSL ONLY
# Usa o melhor modelo já observado:
# ssl_wavlm + rbf_svc_balanced = f1_macro 0.6152
# Não precisa de manual_features_v5.csv
# ============================================================

from pathlib import Path
import os
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

# ------------------------------------------------------------
# 1) Procurar automaticamente os ficheiros SSL
# ------------------------------------------------------------

cwd = Path.cwd()
home = Path.home()

search_roots = []

# pasta atual e pais
search_roots.append(cwd)
search_roots.extend(list(cwd.parents))

# home e pastas comuns
for p in [
    home,
    home / "MIA",
    home / "ASLAI--SA",
    home / "Downloads",
    Path("/content/drive/MyDrive"),
]:
    if p.exists():
        search_roots.append(p)

# remover duplicados mantendo ordem
unique_roots = []
seen = set()
for r in search_roots:
    r = r.resolve()
    if r not in seen and r.exists():
        unique_roots.append(r)
        seen.add(r)

target_npz_names = [
    "ssl_embeddings_v5_wavlm_base.npz",
    "ssl_embeddings_wavlm_base.npz",
    "ssl_embeddings.npz",
]

target_meta_names = [
    "ssl_embeddings_v5_wavlm_base_meta.csv",
    "ssl_embeddings_wavlm_base_meta.csv",
    "ssl_meta.csv",
]

def find_first_file(names, roots):
    matches = []

    for root in roots:
        # primeiro: procura direta em prosody_outputs_v5, se existir
        pdir = root / "prosody_outputs_v5"
        if pdir.exists():
            for name in names:
                p = pdir / name
                if p.exists():
                    matches.append(p)

        # depois: procura recursiva limitada
        for name in names:
            try:
                found = list(root.rglob(name))
                matches.extend(found)
            except Exception:
                pass

    # remover duplicados
    clean = []
    seen = set()
    for m in matches:
        m = m.resolve()
        if m not in seen:
            clean.append(m)
            seen.add(m)

    return clean

npz_matches = find_first_file(target_npz_names, unique_roots)
meta_matches = find_first_file(target_meta_names, unique_roots)

print("NPZ encontrados:")
for p in npz_matches[:10]:
    print("-", p)

print("\nMETA encontrados:")
for p in meta_matches[:10]:
    print("-", p)

if not npz_matches:
    raise FileNotFoundError(
        "Não encontrei o ficheiro ssl_embeddings_v5_wavlm_base.npz. "
        "Tens de localizar a pasta prosody_outputs_v5 onde ele foi gerado."
    )

if not meta_matches:
    raise FileNotFoundError(
        "Não encontrei o ficheiro ssl_embeddings_v5_wavlm_base_meta.csv. "
        "Tens de localizar a pasta prosody_outputs_v5 onde ele foi gerado."
    )

ssl_npz_path = npz_matches[0]
ssl_meta_path = meta_matches[0]
OUTPUT_DIR = ssl_npz_path.parent

print("\nA usar:")
print("OUTPUT_DIR:", OUTPUT_DIR)
print("ssl_npz_path:", ssl_npz_path)
print("ssl_meta_path:", ssl_meta_path)

# ------------------------------------------------------------
# 2) Carregar embeddings SSL e metadata
# ------------------------------------------------------------

npz = np.load(ssl_npz_path)

print("\nChaves dentro do NPZ:", list(npz.keys()))

if "X_ssl" in npz:
    X_ssl = npz["X_ssl"]
elif "embeddings" in npz:
    X_ssl = npz["embeddings"]
elif len(npz.files) == 1:
    X_ssl = npz[npz.files[0]]
else:
    raise KeyError(f"Não sei qual chave usar no NPZ. Chaves: {npz.files}")

ssl_meta = pd.read_csv(ssl_meta_path)

print("\nShapes:")
print("X_ssl:", X_ssl.shape)
print("ssl_meta:", ssl_meta.shape)
print("Colunas ssl_meta:", ssl_meta.columns.tolist())

required_cols = ["label", "speaker", "source_dataset"]
missing = [c for c in required_cols if c not in ssl_meta.columns]

if missing:
    raise ValueError(
        f"Faltam colunas no ssl_meta: {missing}. "
        f"Colunas existentes: {ssl_meta.columns.tolist()}"
    )

if len(X_ssl) != len(ssl_meta):
    raise ValueError(
        f"Inconsistência: X_ssl tem {len(X_ssl)} linhas, "
        f"mas ssl_meta tem {len(ssl_meta)} linhas."
    )

# ------------------------------------------------------------
# 3) Preparar y, groups, sources
# ------------------------------------------------------------

EMOTIONS = ["joy", "sadness", "surprise", "fear", "anger", "disgust", "neutral"]

y_labels = ssl_meta["label"].astype(str).to_numpy()
groups = ssl_meta["speaker"].astype(str).to_numpy()
sources = ssl_meta["source_dataset"].astype(str).to_numpy()

unknown = sorted(set(y_labels) - set(EMOTIONS))
if unknown:
    raise ValueError(f"Labels desconhecidas encontradas: {unknown}")

label_to_idx = {e: i for i, e in enumerate(EMOTIONS)}
idx_to_label = {i: e for e, i in label_to_idx.items()}

y = np.array([label_to_idx[l] for l in y_labels], dtype=int)

print("\nDistribuição por emoção:")
display(pd.Series(y_labels).value_counts())

print("\nDistribuição por dataset:")
display(pd.Series(sources).value_counts())

# ------------------------------------------------------------
# 4) Reconstruir tabela de resultados observados
# ------------------------------------------------------------

results_recovered = [
    {"feature_set": "manual", "model": "logreg_balanced", "f1_macro": 0.4778},
    {"feature_set": "manual", "model": "linear_svc_calibrated", "f1_macro": 0.4353},
    {"feature_set": "manual", "model": "rbf_svc_balanced", "f1_macro": 0.5223},
    {"feature_set": "manual", "model": "extra_trees", "f1_macro": 0.4718},
    {"feature_set": "manual", "model": "random_forest", "f1_macro": 0.4665},
    {"feature_set": "manual", "model": "hist_gradient_boosting", "f1_macro": 0.5209},

    {"feature_set": "ssl_wavlm", "model": "logreg_balanced", "f1_macro": 0.5604},
    {"feature_set": "ssl_wavlm", "model": "linear_svc_calibrated", "f1_macro": 0.4969},
    {"feature_set": "ssl_wavlm", "model": "rbf_svc_balanced", "f1_macro": 0.6152},
    {"feature_set": "ssl_wavlm", "model": "extra_trees", "f1_macro": 0.4851},
    {"feature_set": "ssl_wavlm", "model": "random_forest", "f1_macro": 0.4885},
    {"feature_set": "ssl_wavlm", "model": "hist_gradient_boosting", "f1_macro": 0.5456},

    {"feature_set": "manual_plus_ssl", "model": "logreg_balanced", "f1_macro": 0.5770},
]

results_df = pd.DataFrame(results_recovered).sort_values(
    "f1_macro",
    ascending=False
).reset_index(drop=True)

results_path = OUTPUT_DIR / "prosody_model_benchmark_v5_RESUME_NO_REPAIR_RECOVERED_SSL_ONLY.csv"
results_df.to_csv(results_path, index=False)

print("\nResultados recuperados:")
display(results_df)

print("Resultados guardados em:", results_path)

# ------------------------------------------------------------
# 5) Treinar modelo final SSL/WavLM + RBF SVC
# ------------------------------------------------------------

RANDOM_STATE = 42

final_model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(
        C=3.0,
        gamma="scale",
        kernel="rbf",
        class_weight="balanced",
        probability=True,
        random_state=RANDOM_STATE
    ))
])

print("\nA treinar modelo final SSL/WavLM + RBF SVC no dataset completo...")
print("Isto pode demorar alguns minutos, mas é muito menos do que repetir o benchmark todo.")

final_model.fit(X_ssl.astype(np.float32), y)

print("Modelo final treinado com sucesso.")

# ------------------------------------------------------------
# 6) Guardar artefacto final
# ------------------------------------------------------------

artifact = {
    "model": final_model,
    "feature_set": "ssl_wavlm",
    "model_name": "rbf_svc_balanced",
    "emotions": EMOTIONS,
    "label_to_idx": label_to_idx,
    "idx_to_label": idx_to_label,
    "results": results_df,
    "selected_strategy": "ssl_wavlm",
    "final_model_name": "rbf_svc_balanced",
    "ssl_model_name": "microsoft/wavlm-base",
    "ssl_sample_rate": 16000,
    "ssl_embedding_dim": int(X_ssl.shape[1]),
    "uses_manual_features": False,
    "note": (
        "Artefacto recuperado em modo SSL only. "
        "Usa embeddings WavLM já extraídos e modelo RBF SVC. "
        "Resultados CV foram recuperados do output visível do notebook "
        "porque o kernel foi perdido após desligar o PC."
    )
}

artifact_path = OUTPUT_DIR / "prosody_final_artifact_v5_SSL_ONLY_RECOVERED.joblib"
joblib.dump(artifact, artifact_path)

summary = {
    "best_model": {
        "feature_set": "ssl_wavlm",
        "model": "rbf_svc_balanced",
        "f1_macro_cv_observed": 0.6152
    },
    "n_samples": int(len(y)),
    "n_speakers": int(len(np.unique(groups))),
    "class_distribution": pd.Series(y_labels).value_counts().to_dict(),
    "source_distribution": pd.Series(sources).value_counts().to_dict(),
    "artifact_path": str(artifact_path),
    "warning": "Sem ensemble OOF porque o kernel foi perdido/desligado."
}

summary_path = OUTPUT_DIR / "summary_v5_SSL_ONLY_RECOVERED.json"
summary_path.write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("\n============================================")
print("ARTEFACTO FINAL GUARDADO")
print("============================================")
print(artifact_path)

print("\nResumo guardado em:")
print(summary_path)

print("\nResumo:")
print(json.dumps(summary, ensure_ascii=False, indent=2))

NPZ encontrados:

META encontrados:


FileNotFoundError: Não encontrei o ficheiro ssl_embeddings_v5_wavlm_base.npz. Tens de localizar a pasta prosody_outputs_v5 onde ele foi gerado.

In [7]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Drive montado.")
except Exception as e:
    print("Não consegui montar com google.colab.drive.")
    print("Erro:", repr(e))

Mounted at /content/drive
Drive montado.


In [8]:
from pathlib import Path

roots = [
    Path("/content/drive/MyDrive"),
    Path("/content/drive/Shareddrives"),
]

found = []

for root in roots:
    if root.exists():
        print("A procurar em:", root)
        for p in root.rglob("prosody_outputs_v5"):
            if p.is_dir():
                found.append(p)
                print("ENCONTRADO:", p)

if not found:
    raise FileNotFoundError("Não encontrei prosody_outputs_v5 no Drive montado.")
else:
    OUTPUT_DIR = found[0]
    print("\nOUTPUT_DIR definido como:")
    print(OUTPUT_DIR)

A procurar em: /content/drive/MyDrive
ENCONTRADO: /content/drive/MyDrive/prosody_outputs_v5

OUTPUT_DIR definido como:
/content/drive/MyDrive/prosody_outputs_v5


In [9]:
print("Conteúdo de:", OUTPUT_DIR)

for p in sorted(OUTPUT_DIR.iterdir()):
    print(p.name)

Conteúdo de: /content/drive/MyDrive/prosody_outputs_v5
class_priors_v5.json
manual_feature_columns_v5.json
manual_features_v5.csv
metadata_v5_clean.csv
ssl_embedding_failures_v5.csv
ssl_embeddings_v5_wavlm_base.npz
ssl_embeddings_v5_wavlm_base_meta.csv
ssl_feature_columns_v5.json


In [10]:
from pathlib import Path

base = Path("/content/drive/MyDrive")

found = list(base.rglob("prosody_outputs_v5"))

print("Pastas encontradas:")
for p in found:
    print(p)

if not found:
    raise FileNotFoundError("Não encontrei prosody_outputs_v5 dentro do MyDrive.")
else:
    OUTPUT_DIR = found[0]
    print("\nOUTPUT_DIR definido como:")
    print(OUTPUT_DIR)

Pastas encontradas:
/content/drive/MyDrive/prosody_outputs_v5

OUTPUT_DIR definido como:
/content/drive/MyDrive/prosody_outputs_v5


In [11]:
print("Conteúdo de:", OUTPUT_DIR)

for p in sorted(OUTPUT_DIR.iterdir()):
    print(p.name)

Conteúdo de: /content/drive/MyDrive/prosody_outputs_v5
class_priors_v5.json
manual_feature_columns_v5.json
manual_features_v5.csv
metadata_v5_clean.csv
ssl_embedding_failures_v5.csv
ssl_embeddings_v5_wavlm_base.npz
ssl_embeddings_v5_wavlm_base_meta.csv
ssl_feature_columns_v5.json


In [12]:
# ============================================================
# FINAL RECOVERY — treinar modelo final de prosódia com SSL/WavLM
# Usa a cache já existente em prosody_outputs_v5
# Melhor resultado observado:
# ssl_wavlm + rbf_svc_balanced = f1_macro 0.6152
# ============================================================

from pathlib import Path
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import joblib

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

# ------------------------------------------------------------
# 1) Garantir OUTPUT_DIR correto
# ------------------------------------------------------------

OUTPUT_DIR = Path("/content/drive/MyDrive/prosody_outputs_v5")

if not OUTPUT_DIR.exists():
    raise FileNotFoundError(f"Não encontrei OUTPUT_DIR: {OUTPUT_DIR}")

print("OUTPUT_DIR:", OUTPUT_DIR)

ssl_npz_path = OUTPUT_DIR / "ssl_embeddings_v5_wavlm_base.npz"
ssl_meta_path = OUTPUT_DIR / "ssl_embeddings_v5_wavlm_base_meta.csv"

if not ssl_npz_path.exists():
    raise FileNotFoundError(f"Não encontrei: {ssl_npz_path}")

if not ssl_meta_path.exists():
    raise FileNotFoundError(f"Não encontrei: {ssl_meta_path}")

print("NPZ:", ssl_npz_path.name)
print("META:", ssl_meta_path.name)

# ------------------------------------------------------------
# 2) Carregar embeddings SSL/WavLM e metadata
# ------------------------------------------------------------

npz = np.load(ssl_npz_path)
print("Chaves no NPZ:", list(npz.keys()))

if "X_ssl" in npz:
    X_ssl = npz["X_ssl"]
elif "embeddings" in npz:
    X_ssl = npz["embeddings"]
elif len(npz.files) == 1:
    X_ssl = npz[npz.files[0]]
else:
    raise KeyError(f"Não sei qual chave usar no NPZ. Chaves: {npz.files}")

ssl_meta = pd.read_csv(ssl_meta_path)

print("X_ssl shape:", X_ssl.shape)
print("ssl_meta shape:", ssl_meta.shape)
print("Colunas ssl_meta:", ssl_meta.columns.tolist())

# ------------------------------------------------------------
# 3) Preparar labels
# ------------------------------------------------------------

EMOTIONS = ["joy", "sadness", "surprise", "fear", "anger", "disgust", "neutral"]

if "label" not in ssl_meta.columns:
    raise ValueError("A coluna 'label' não existe no ssl_meta.")

y_labels = ssl_meta["label"].astype(str).to_numpy()

unknown = sorted(set(y_labels) - set(EMOTIONS))
if unknown:
    raise ValueError(f"Labels desconhecidas encontradas: {unknown}")

label_to_idx = {e: i for i, e in enumerate(EMOTIONS)}
idx_to_label = {i: e for e, i in label_to_idx.items()}

y = np.array([label_to_idx[label] for label in y_labels], dtype=int)

if len(X_ssl) != len(y):
    raise ValueError(
        f"X_ssl tem {len(X_ssl)} amostras, mas y tem {len(y)} labels."
    )

print("\nDistribuição por emoção:")
display(pd.Series(y_labels).value_counts())

# ------------------------------------------------------------
# 4) Guardar tabela de resultados recuperados
# ------------------------------------------------------------

results_recovered = [
    {"feature_set": "manual", "model": "logreg_balanced", "f1_macro": 0.4778},
    {"feature_set": "manual", "model": "linear_svc_calibrated", "f1_macro": 0.4353},
    {"feature_set": "manual", "model": "rbf_svc_balanced", "f1_macro": 0.5223},
    {"feature_set": "manual", "model": "extra_trees", "f1_macro": 0.4718},
    {"feature_set": "manual", "model": "random_forest", "f1_macro": 0.4665},
    {"feature_set": "manual", "model": "hist_gradient_boosting", "f1_macro": 0.5209},

    {"feature_set": "ssl_wavlm", "model": "logreg_balanced", "f1_macro": 0.5604},
    {"feature_set": "ssl_wavlm", "model": "linear_svc_calibrated", "f1_macro": 0.4969},
    {"feature_set": "ssl_wavlm", "model": "rbf_svc_balanced", "f1_macro": 0.6152},
    {"feature_set": "ssl_wavlm", "model": "extra_trees", "f1_macro": 0.4851},
    {"feature_set": "ssl_wavlm", "model": "random_forest", "f1_macro": 0.4885},
    {"feature_set": "ssl_wavlm", "model": "hist_gradient_boosting", "f1_macro": 0.5456},

    {"feature_set": "manual_plus_ssl", "model": "logreg_balanced", "f1_macro": 0.5770},
]

results_df = pd.DataFrame(results_recovered).sort_values(
    "f1_macro",
    ascending=False
).reset_index(drop=True)

results_path = OUTPUT_DIR / "prosody_model_benchmark_v5_RECOVERED.csv"
results_df.to_csv(results_path, index=False)

print("\nResultados recuperados:")
display(results_df)

print("Resultados guardados em:", results_path)

# ------------------------------------------------------------
# 5) Treinar modelo final no dataset todo
# ------------------------------------------------------------

RANDOM_STATE = 42

final_model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(
        C=3.0,
        gamma="scale",
        kernel="rbf",
        class_weight="balanced",
        probability=True,
        random_state=RANDOM_STATE
    ))
])

print("\nA treinar modelo final: ssl_wavlm + rbf_svc_balanced")
print("Pode demorar alguns minutos. Espera até terminar.")

final_model.fit(X_ssl.astype(np.float32), y)

print("Modelo final treinado com sucesso.")

# ------------------------------------------------------------
# 6) Guardar artefacto final para usar no late_fusion.ipynb
# ------------------------------------------------------------

artifact = {
    "model": final_model,
    "feature_set": "ssl_wavlm",
    "model_name": "rbf_svc_balanced",
    "selected_strategy": "ssl_wavlm",
    "final_model_name": "rbf_svc_balanced",

    "emotions": EMOTIONS,
    "label_to_idx": label_to_idx,
    "idx_to_label": idx_to_label,

    "ssl_model_name": "microsoft/wavlm-base",
    "ssl_sample_rate": 16000,
    "ssl_embedding_dim": int(X_ssl.shape[1]),

    "uses_manual_features": False,
    "results": results_df,

    "note": (
        "Artefacto final de prosódia recuperado após perda do estado do kernel. "
        "Usa apenas embeddings SSL/WavLM, porque foi o melhor feature set observado. "
        "Melhor resultado observado em validação cruzada: ssl_wavlm + rbf_svc_balanced, f1_macro=0.6152."
    )
}

artifact_path = OUTPUT_DIR / "prosody_final_artifact_v5_SSL_WAVLM_RBF_RECOVERED.joblib"
joblib.dump(artifact, artifact_path)

summary = {
    "best_model": {
        "feature_set": "ssl_wavlm",
        "model": "rbf_svc_balanced",
        "f1_macro_cv_observed": 0.6152
    },
    "n_samples": int(len(y)),
    "class_distribution": pd.Series(y_labels).value_counts().to_dict(),
    "artifact_path": str(artifact_path),
    "warning": "Ensemble OOF não usado porque o kernel foi desligado e o oof_store perdeu-se."
}

summary_path = OUTPUT_DIR / "summary_v5_SSL_WAVLM_RBF_RECOVERED.json"
summary_path.write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("\n============================================")
print("ARTEFACTO FINAL GUARDADO")
print("============================================")
print(artifact_path)

print("\nResumo guardado em:")
print(summary_path)

print("\nResumo:")
print(json.dumps(summary, ensure_ascii=False, indent=2))

OUTPUT_DIR: /content/drive/MyDrive/prosody_outputs_v5
NPZ: ssl_embeddings_v5_wavlm_base.npz
META: ssl_embeddings_v5_wavlm_base_meta.csv
Chaves no NPZ: ['X_ssl']
X_ssl shape: (12788, 1536)
ssl_meta shape: (12788, 4)
Colunas ssl_meta: ['path', 'label', 'speaker', 'source_dataset']

Distribuição por emoção:


,count
anger,2167
joy,2167
sadness,2166
fear,2047
disgust,1863
neutral,1786
surprise,592



Resultados recuperados:


,feature_set,model,f1_macro
0,ssl_wavlm,rbf_svc_balanced,0.6152
1,manual_plus_ssl,logreg_balanced,0.5770
2,ssl_wavlm,logreg_balanced,0.5604
3,ssl_wavlm,hist_gradient_boosting,0.5456
4,manual,rbf_svc_balanced,0.5223
5,manual,hist_gradient_boosting,0.5209
6,ssl_wavlm,linear_svc_calibrated,0.4969
7,ssl_wavlm,random_forest,0.4885
8,ssl_wavlm,extra_trees,0.4851
9,manual,logreg_balanced,0.4778


Resultados guardados em: /content/drive/MyDrive/prosody_outputs_v5/prosody_model_benchmark_v5_RECOVERED.csv

A treinar modelo final: ssl_wavlm + rbf_svc_balanced
Pode demorar alguns minutos. Espera até terminar.
Modelo final treinado com sucesso.

ARTEFACTO FINAL GUARDADO
/content/drive/MyDrive/prosody_outputs_v5/prosody_final_artifact_v5_SSL_WAVLM_RBF_RECOVERED.joblib

Resumo guardado em:
/content/drive/MyDrive/prosody_outputs_v5/summary_v5_SSL_WAVLM_RBF_RECOVERED.json

Resumo:
{
  "best_model": {
    "feature_set": "ssl_wavlm",
    "model": "rbf_svc_balanced",
    "f1_macro_cv_observed": 0.6152
  },
  "n_samples": 12788,
  "class_distribution": {
    "anger": 2167,
    "joy": 2167,
    "sadness": 2166,
    "fear": 2047,
    "disgust": 1863,
    "neutral": 1786,
    "surprise": 592
  },
  "artifact_path": "/content/drive/MyDrive/prosody_outputs_v5/prosody_final_artifact_v5_SSL_WAVLM_RBF_RECOVERED.joblib",
  "warning": "Ensemble OOF não usado porque o kernel foi desligado e o oof_store

In [13]:
from pathlib import Path
import joblib

artifact_path = Path("/content/drive/MyDrive/prosody_outputs_v5/prosody_final_artifact_v5_SSL_WAVLM_RBF_RECOVERED.joblib")

print("Existe?", artifact_path.exists())
print("Tamanho MB:", round(artifact_path.stat().st_size / 1024 / 1024, 2) if artifact_path.exists() else None)

art = joblib.load(artifact_path)

print("Feature set:", art["feature_set"])
print("Modelo:", art["model_name"])
print("Emoções:", art["emotions"])
print("Dimensão SSL:", art["ssl_embedding_dim"])

Existe? True
Tamanho MB: 122.1
Feature set: ssl_wavlm
Modelo: rbf_svc_balanced
Emoções: ['joy', 'sadness', 'surprise', 'fear', 'anger', 'disgust', 'neutral']
Dimensão SSL: 1536


## 8) Ensemble simples dos melhores modelos

Faz média das probabilidades OOF dos melhores modelos.


In [3]:
valid_results = results_df.dropna(subset=["f1_macro"]).copy()
top_k = min(5, len(valid_results))

ensemble_rows = []

if top_k >= 2:
    top_models = list(valid_results.head(top_k)[["feature_set", "model"]].itertuples(index=False, name=None))
    print("Modelos no ensemble:")
    for m in top_models:
        print("-", m)

    avg_proba = np.mean([oof_store[m]["proba"] for m in top_models], axis=0)
    ens_pred = avg_proba.argmax(axis=1)

    ens_row = {
        "feature_set": "ensemble_top_models",
        "model": f"avg_proba_top_{top_k}",
        "accuracy": accuracy_score(y, ens_pred),
        "balanced_accuracy": balanced_accuracy_score(y, ens_pred),
        "f1_macro": f1_score(y, ens_pred, average="macro"),
        "f1_weighted": f1_score(y, ens_pred, average="weighted"),
        "elapsed_sec": np.nan,
    }
    ensemble_rows.append(ens_row)

    results_with_ensemble = pd.concat([results_df, pd.DataFrame(ensemble_rows)], ignore_index=True)
    results_with_ensemble = results_with_ensemble.sort_values("f1_macro", ascending=False)
else:
    print("Poucos modelos válidos para ensemble.")
    results_with_ensemble = results_df.copy()
    ens_pred = None

display(results_with_ensemble)

out_path = OUTPUT_DIR / "prosody_model_benchmark_v5_RESUME_NO_REPAIR_with_ensemble.csv"
results_with_ensemble.to_csv(out_path, index=False)
print("Guardado:", out_path)

NameError: name 'results_df' is not defined

In [4]:
# ============================================================
# 7B_RECOVER. Recuperar estado antes da secção 8
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import joblib

# ------------------------------------------------------------
# 1) Encontrar ou definir OUTPUT_DIR
# ------------------------------------------------------------

if "OUTPUT_DIR" in globals():
    OUTPUT_DIR = Path(OUTPUT_DIR)
else:
    cwd = Path.cwd()
    found = []

    for base in [cwd] + list(cwd.parents):
        p = base / "prosody_outputs_v5"
        if p.exists():
            found.append(p)

    if found:
        OUTPUT_DIR = found[0]
        print("OUTPUT_DIR encontrado:", OUTPUT_DIR)
    else:
        OUTPUT_DIR = cwd / "prosody_outputs_v5"
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        print("OUTPUT_DIR criado:", OUTPUT_DIR)

# ------------------------------------------------------------
# 2) Tentar criar results_df
# ------------------------------------------------------------

CHECKPOINT_PATH = OUTPUT_DIR / "prosody_resume_checkpoint_v5_NO_REPAIR.joblib"
PARTIAL_RESULTS_PATH = OUTPUT_DIR / "prosody_model_benchmark_v5_RESUME_NO_REPAIR_PARTIAL.csv"
FULL_RESULTS_PATH = OUTPUT_DIR / "prosody_model_benchmark_v5_RESUME_NO_REPAIR.csv"

loaded_from = None

# Caso A: ainda tens a lista results em memória
if "results" in globals() and len(results) > 0:
    print("A recuperar results_df a partir da variável results em memória.")
    results_df = pd.DataFrame(results)
    loaded_from = "memoria_results"

# Caso B: existe checkpoint
elif CHECKPOINT_PATH.exists():
    print("A carregar checkpoint:", CHECKPOINT_PATH)
    ckpt = joblib.load(CHECKPOINT_PATH)

    results = ckpt.get("results", [])
    fold_tables = ckpt.get("fold_tables", [])
    oof_store = ckpt.get("oof_store", {})

    results_df = pd.DataFrame(results)
    loaded_from = "checkpoint"

# Caso C: existe CSV parcial
elif PARTIAL_RESULTS_PATH.exists():
    print("A carregar CSV parcial:", PARTIAL_RESULTS_PATH)
    results_df = pd.read_csv(PARTIAL_RESULTS_PATH)
    loaded_from = "csv_parcial"

# Caso D: existe CSV final
elif FULL_RESULTS_PATH.exists():
    print("A carregar CSV final:", FULL_RESULTS_PATH)
    results_df = pd.read_csv(FULL_RESULTS_PATH)
    loaded_from = "csv_final"

else:
    raise RuntimeError(
        "Não consegui recuperar results_df. "
        "Não há variável results em memória, nem checkpoint, nem CSV guardado."
    )

# ------------------------------------------------------------
# 3) Ordenar e mostrar
# ------------------------------------------------------------

if "f1_macro" not in results_df.columns:
    raise RuntimeError("O results_df existe, mas não tem coluna f1_macro.")

results_df = results_df.sort_values(
    "f1_macro",
    ascending=False,
    na_position="last"
).reset_index(drop=True)

print("\nresults_df recuperado de:", loaded_from)
print("Shape:", results_df.shape)

display(results_df)

# ------------------------------------------------------------
# 4) Verificar se dá para fazer ensemble
# ------------------------------------------------------------

if "oof_store" in globals() and isinstance(oof_store, dict) and len(oof_store) > 0:
    print("\nOK: oof_store existe. Podes correr a secção 8) Ensemble.")
    print("Nº de modelos com OOF guardado:", len(oof_store))
else:
    print("\nATENÇÃO: não encontrei oof_store.")
    print("Podes ter results_df, mas a secção 8) Ensemble pode falhar.")
    print("Nesse caso, salta o ensemble e treina o modelo final diretamente com o melhor resultado.")

OUTPUT_DIR criado: /content/prosody_outputs_v5


RuntimeError: Não consegui recuperar results_df. Não há variável results em memória, nem checkpoint, nem CSV guardado.

In [5]:
# ============================================================
# RECOVERY FINAL — usar resultados visíveis e treinar artefacto final
# Não precisa de voltar a correr o benchmark inteiro
# ============================================================

from pathlib import Path
import os
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier

# ------------------------------------------------------------
# 1) Encontrar a pasta prosody_outputs_v5
# ------------------------------------------------------------

cwd = Path.cwd()
candidates = []

# procura em cwd e nos pais
for base in [cwd] + list(cwd.parents):
    p = base / "prosody_outputs_v5"
    if p.exists():
        candidates.append(p)

# caso estejas em Colab
colab_path = Path("/content/drive/MyDrive/prosody_outputs_v5")
if colab_path.exists():
    candidates.append(colab_path)

if not candidates:
    raise FileNotFoundError(
        "Não encontrei a pasta prosody_outputs_v5. "
        "Confirma onde ela está e define OUTPUT_DIR manualmente."
    )

OUTPUT_DIR = candidates[0]
print("OUTPUT_DIR:", OUTPUT_DIR)

# ------------------------------------------------------------
# 2) Função para escolher ficheiros dentro da pasta
# ------------------------------------------------------------

def pick_file(candidates, glob_patterns=None, required=True):
    for name in candidates:
        p = OUTPUT_DIR / name
        if p.exists():
            return p

    if glob_patterns:
        matches = []
        for pat in glob_patterns:
            matches.extend(sorted(OUTPUT_DIR.glob(pat)))
        if matches:
            return matches[0]

    if required:
        raise FileNotFoundError(
            f"Não encontrei nenhum destes ficheiros: {candidates} / {glob_patterns}"
        )

    return None

manual_path = pick_file(
    ["manual_features_v5.csv", "manual_features.csv"],
    ["*manual*features*.csv", "*features*v5*.csv"]
)

manual_cols_path = pick_file(
    ["manual_feature_columns_v5.json", "manual_feature_cols_v5.json", "manual_feature_columns.json"],
    ["*manual*column*.json", "*feature*col*.json"]
)

ssl_npz_path = pick_file(
    ["ssl_embeddings_v5_wavlm_base.npz", "ssl_embeddings_wavlm_base.npz", "ssl_embeddings.npz"],
    ["*ssl*wavlm*.npz", "*embedding*.npz"]
)

ssl_meta_path = pick_file(
    ["ssl_embeddings_v5_wavlm_base_meta.csv", "ssl_embeddings_wavlm_base_meta.csv", "ssl_meta.csv"],
    ["*ssl*meta*.csv", "*embedding*meta*.csv"]
)

print("\nFicheiros usados:")
print("manual_path:", manual_path.name)
print("manual_cols_path:", manual_cols_path.name)
print("ssl_npz_path:", ssl_npz_path.name)
print("ssl_meta_path:", ssl_meta_path.name)

# ------------------------------------------------------------
# 3) Carregar features manuais e embeddings SSL já existentes
# ------------------------------------------------------------

with open(manual_cols_path, "r", encoding="utf-8") as f:
    manual_feature_cols = json.load(f)

manual_df = pd.read_csv(manual_path)
ssl_meta = pd.read_csv(ssl_meta_path)

npz = np.load(ssl_npz_path)

if "X_ssl" in npz:
    X_ssl = npz["X_ssl"]
elif "embeddings" in npz:
    X_ssl = npz["embeddings"]
elif len(npz.files) == 1:
    X_ssl = npz[npz.files[0]]
else:
    raise KeyError(f"Não sei qual chave usar no NPZ. Chaves: {npz.files}")

print("\nShapes:")
print("manual_df:", manual_df.shape)
print("ssl_meta:", ssl_meta.shape)
print("X_ssl:", X_ssl.shape)
print("n manual_feature_cols:", len(manual_feature_cols))

# ------------------------------------------------------------
# 4) Alinhar tabelas
# ------------------------------------------------------------

EMOTIONS = ["joy", "sadness", "surprise", "fear", "anger", "disgust", "neutral"]

base_cols = ["path", "filename", "label", "speaker", "source_dataset"]

missing = [c for c in base_cols if c not in manual_df.columns]
if missing:
    raise ValueError(f"Faltam colunas em manual_df: {missing}")

data = manual_df[base_cols].drop_duplicates("path").copy()

feature_table = manual_df[["path"] + manual_feature_cols].drop_duplicates("path")
data = data.merge(feature_table, on="path", how="inner")

sm = ssl_meta.copy()
if "ssl_idx" not in sm.columns:
    sm = sm.reset_index().rename(columns={"index": "ssl_idx"})

if "path" not in sm.columns:
    raise ValueError(f"ssl_meta não tem coluna 'path'. Colunas: {sm.columns.tolist()}")

data = data.merge(
    sm[["path", "ssl_idx"]].drop_duplicates("path"),
    on="path",
    how="inner"
)

X_manual = data[manual_feature_cols].to_numpy(dtype=np.float32)
X_ssl_aligned = X_ssl[data["ssl_idx"].to_numpy()].astype(np.float32)

y_labels = data["label"].astype(str).to_numpy()
groups = data["speaker"].astype(str).to_numpy()
sources = data["source_dataset"].astype(str).to_numpy()

unknown = sorted(set(y_labels) - set(EMOTIONS))
if unknown:
    raise ValueError(f"Labels desconhecidas: {unknown}")

label_to_idx = {e: i for i, e in enumerate(EMOTIONS)}
idx_to_label = {i: e for e, i in label_to_idx.items()}
y = np.array([label_to_idx[l] for l in y_labels], dtype=int)

feature_matrices = {
    "manual": X_manual,
    "ssl_wavlm": X_ssl_aligned,
    "manual_plus_ssl": np.hstack([X_manual, X_ssl_aligned]).astype(np.float32),
}

print("\nFeature matrices:")
for name, X in feature_matrices.items():
    print(name, X.shape)

print("\nDistribuição por emoção:")
print(pd.Series(y_labels).value_counts())

# ------------------------------------------------------------
# 5) Definir modelos
# ------------------------------------------------------------

RANDOM_STATE = 42

def make_models():
    return {
        "logreg_balanced": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                max_iter=4000,
                class_weight="balanced",
                C=1.0,
                solver="lbfgs",
                multi_class="auto",
                random_state=RANDOM_STATE
            ))
        ]),
        "linear_svc_calibrated": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", CalibratedClassifierCV(
                estimator=LinearSVC(
                    C=0.5,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                    max_iter=5000
                ),
                cv=3
            ))
        ]),
        "rbf_svc_balanced": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", SVC(
                C=3.0,
                gamma="scale",
                kernel="rbf",
                class_weight="balanced",
                probability=True,
                random_state=RANDOM_STATE
            ))
        ]),
        "extra_trees": ExtraTreesClassifier(
            n_estimators=600,
            max_features="sqrt",
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        "random_forest": RandomForestClassifier(
            n_estimators=500,
            max_features="sqrt",
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        "hist_gradient_boosting": HistGradientBoostingClassifier(
            max_iter=300,
            learning_rate=0.05,
            random_state=RANDOM_STATE
        )
    }

# ------------------------------------------------------------
# 6) Reconstruir results_df a partir dos resultados visíveis
# ------------------------------------------------------------

results_recovered = [
    {"feature_set": "manual", "model": "logreg_balanced", "f1_macro": 0.4778},
    {"feature_set": "manual", "model": "linear_svc_calibrated", "f1_macro": 0.4353},
    {"feature_set": "manual", "model": "rbf_svc_balanced", "f1_macro": 0.5223},
    {"feature_set": "manual", "model": "extra_trees", "f1_macro": 0.4718},
    {"feature_set": "manual", "model": "random_forest", "f1_macro": 0.4665},
    {"feature_set": "manual", "model": "hist_gradient_boosting", "f1_macro": 0.5209},

    {"feature_set": "ssl_wavlm", "model": "logreg_balanced", "f1_macro": 0.5604},
    {"feature_set": "ssl_wavlm", "model": "linear_svc_calibrated", "f1_macro": 0.4969},
    {"feature_set": "ssl_wavlm", "model": "rbf_svc_balanced", "f1_macro": 0.6152},
    {"feature_set": "ssl_wavlm", "model": "extra_trees", "f1_macro": 0.4851},
    {"feature_set": "ssl_wavlm", "model": "random_forest", "f1_macro": 0.4885},
    {"feature_set": "ssl_wavlm", "model": "hist_gradient_boosting", "f1_macro": 0.5456},

    {"feature_set": "manual_plus_ssl", "model": "logreg_balanced", "f1_macro": 0.5770},
]

results_df = pd.DataFrame(results_recovered)

for col in ["accuracy", "balanced_accuracy", "f1_weighted", "elapsed_sec"]:
    if col not in results_df.columns:
        results_df[col] = np.nan

results_df = results_df.sort_values(
    "f1_macro",
    ascending=False,
    na_position="last"
).reset_index(drop=True)

results_with_ensemble = results_df.copy()

results_path = OUTPUT_DIR / "prosody_model_benchmark_v5_RESUME_NO_REPAIR_RECOVERED.csv"
results_df.to_csv(results_path, index=False)

print("\nResultados recuperados guardados em:", results_path)
display(results_df)

# ------------------------------------------------------------
# 7) Treinar o melhor modelo individual no dataset todo
# ------------------------------------------------------------

best_ind = results_df.dropna(subset=["f1_macro"]).iloc[0]

best_fs = best_ind["feature_set"]
best_model_name = best_ind["model"]

print("\nMelhor modelo individual recuperado:")
print("Feature set:", best_fs)
print("Modelo:", best_model_name)
print("F1 macro CV:", best_ind["f1_macro"])

X_best = feature_matrices[best_fs]

final_model = make_models()[best_model_name]

print("\nA treinar modelo final no dataset completo...")
final_model.fit(X_best, y)
print("Modelo final treinado.")

artifact = {
    "model": final_model,
    "feature_set": best_fs,
    "model_name": best_model_name,
    "emotions": EMOTIONS,
    "label_to_idx": label_to_idx,
    "idx_to_label": idx_to_label,
    "manual_feature_cols": manual_feature_cols,
    "results": results_df,
    "selected_strategy": best_fs,
    "final_model_name": best_model_name,
    "ssl_model_name": "microsoft/wavlm-base",
    "ssl_sample_rate": 16000,
    "note": (
        "Modelo final treinado a partir da cache prosody_outputs_v5. "
        "Resultados CV recuperados manualmente a partir do output visível do notebook, "
        "porque o kernel foi reiniciado/desligado antes de guardar o checkpoint."
    )
}

artifact_path = OUTPUT_DIR / "prosody_final_artifact_v5_RESUME_NO_REPAIR_RECOVERED.joblib"
joblib.dump(artifact, artifact_path)

summary = {
    "best_individual_model": best_ind.to_dict(),
    "n_samples": int(len(y)),
    "n_speakers": int(len(np.unique(groups))),
    "class_distribution": pd.Series(y_labels).value_counts().to_dict(),
    "source_distribution": pd.Series(sources).value_counts().to_dict(),
    "artifact_path": str(artifact_path),
    "warning": "Sem ensemble OOF porque o kernel foi reiniciado e o oof_store perdeu-se."
}

summary_path = OUTPUT_DIR / "summary_v5_RESUME_NO_REPAIR_RECOVERED.json"
summary_path.write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("\nArtefacto guardado em:")
print(artifact_path)

print("\nResumo guardado em:")
print(summary_path)

print("\nResumo:")
print(json.dumps(summary, ensure_ascii=False, indent=2))

OUTPUT_DIR: /content/prosody_outputs_v5


FileNotFoundError: Não encontrei nenhum destes ficheiros: ['manual_features_v5.csv', 'manual_features.csv'] / ['*manual*features*.csv', '*features*v5*.csv']

## 9) Relatório do melhor resultado

Mostra a classificação por emoção e a matriz de confusão.


In [ ]:
best = results_with_ensemble.iloc[0]
print("Melhor resultado:")
display(best.to_frame().T)

if best["feature_set"] == "ensemble_top_models":
    best_pred = ens_pred
    best_name = best["model"]
else:
    key = (best["feature_set"], best["model"])
    best_pred = oof_store[key]["pred"]
    best_name = f"{best['feature_set']} / {best['model']}"

target_names = EMOTIONS

report = classification_report(
    y,
    best_pred,
    target_names=target_names,
    digits=4,
    zero_division=0
)
print(report)

report_path = OUTPUT_DIR / "best_classification_report_v5_RESUME_NO_REPAIR.txt"
report_path.write_text(f"BEST: {best_name}\n\n{report}", encoding="utf-8")
print("Guardado:", report_path)

cm = confusion_matrix(y, best_pred, labels=list(range(len(EMOTIONS))))

plt.figure(figsize=(8, 7))
plt.imshow(cm)
plt.title(f"Confusion matrix — {best_name}")
plt.xlabel("Predito")
plt.ylabel("Real")
plt.xticks(range(len(EMOTIONS)), EMOTIONS, rotation=45, ha="right")
plt.yticks(range(len(EMOTIONS)), EMOTIONS)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, str(cm[i, j]), ha="center", va="center")
plt.tight_layout()

cm_path = OUTPUT_DIR / "best_confusion_matrix_v5_RESUME_NO_REPAIR.png"
plt.savefig(cm_path, dpi=200, bbox_inches="tight")
plt.show()

print("Guardado:", cm_path)

## 10) Treinar artefacto final no dataset todo

Treina o melhor modelo individual no conjunto todo.  
Se o melhor resultado for ensemble, treina o melhor modelo individual da tabela original.


In [ ]:
valid_individual = results_df.dropna(subset=["f1_macro"]).copy()
best_ind = valid_individual.iloc[0]

best_fs = best_ind["feature_set"]
best_model_name = best_ind["model"]
X_best = feature_matrices[best_fs]

final_model = make_models()[best_model_name]
final_model.fit(X_best, y)

artifact = {
    "model": final_model,
    "feature_set": best_fs,
    "model_name": best_model_name,
    "emotions": EMOTIONS,
    "label_to_idx": label_to_idx,
    "idx_to_label": idx_to_label,
    "manual_feature_cols": manual_feature_cols,
    "results": results_with_ensemble,
    "note": "Modelo treinado a partir de cache prosody_outputs_v5, sem voltar a correr WavLM."
}

artifact_path = OUTPUT_DIR / "prosody_final_artifact_v5_RESUME_NO_REPAIR.joblib"
joblib.dump(artifact, artifact_path)

summary = {
    "best_cv_result": best.to_dict(),
    "best_individual_model": best_ind.to_dict(),
    "n_samples": int(len(y)),
    "n_speakers": int(len(np.unique(groups))),
    "class_distribution": pd.Series(y_labels).value_counts().to_dict(),
    "source_distribution": pd.Series(sources).value_counts().to_dict(),
    "artifact_path": str(artifact_path),
}

summary_path = OUTPUT_DIR / "summary_v5_RESUME_NO_REPAIR.json"
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

print("Artefacto guardado:", artifact_path)
print("Resumo guardado:", summary_path)
print("\nResumo:")
print(json.dumps(summary, ensure_ascii=False, indent=2))